This is the final notebook for ctxsensitive test results. Rerun in order to obtain the ctxsensitive results from the semester project.

In [1]:
import numpy as np
import json
import sys
from pathlib import Path
import torch
from torch import nn
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from sem_proj.data.datasets import BoasSequenceDataset
from sem_proj.data.preprocessing import PreprocessingConfig
from sem_proj.models.model_factory import SSLEpochTransformerConv1D_v2, SequenceGRUClassifier

CHECKPOINT_LEOMED_DIR = PROJECT_ROOT / "checkpoints_leomed"
JSON_DIR = PROJECT_ROOT / "reports" / "metrics"
TARGET_DIR = PROJECT_ROOT / "plots"
SPLITS_FILE = PROJECT_ROOT / "data" / "processed" / "data_splits_70_15_15.json"
CONFIG_DIR = PROJECT_ROOT / "configs" / "preprocess"

In [2]:
def forwardpass_testset(model, dataloader, device, return_preds=False):
    model.eval()
    total_correct = 0
    total_samples = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:
            x, y = batch
            x = x.to(device)  # (B, L, C, T)
            y = y.to(device)  # (B, L)

            output = model(x)  # (B, L, num_classes)

            preds = output.argmax(dim=-1)  # (B, L)
            total_correct += (preds == y).sum().item()
            total_samples += y.numel()

            all_preds.append(preds.cpu().numpy().flatten())
            all_labels.append(y.cpu().numpy().flatten())
    
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    accuracy = total_correct / total_samples if total_samples > 0 else 0.0

    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    per_class_f1 = f1_score(all_labels, all_preds, average=None, zero_division=0)
    if return_preds:
        return accuracy, macro_f1, per_class_f1, all_labels, all_preds
    return accuracy, macro_f1, per_class_f1

In [3]:
preprocess_config = PreprocessingConfig.from_yaml(CONFIG_DIR / "notch_bandpass_resample_znorm.yaml")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
with open(SPLITS_FILE, 'r') as f:
    splits = json.load(f)
test_nights = splits['test_subjects']

forward pass testset unidir gru L=5 (s=1 for train)

In [4]:
exp = "ctxsensitive_val_step_"
names_finetuned_unidir_l5_s1 = [exp + f"p{p}_ssl_finetuning_bidirFalse_L5_s1" for p in [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]]
names_fullysupervised_unidir_l5_s1 = [exp + f"p{p}_fully_supervised_bidirFalse_L5_s1" for p in [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]]
test_metrics_file_finetuned_unidir_l5_s1 = JSON_DIR / "ctxsensitive_finetuning_results_test_forward_pass_unidir_l5_s1_extra.json"
test_metrics_file_fullysupervised_unidir_l5_s1 = JSON_DIR / "ctxsensitive_fullysupervised_results_test_forward_pass_unidir_l5_s1_extra.json"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [5]:
# forward pass testset for finetuned model
encoder = SSLEpochTransformerConv1D_v2(
    d_model=128,
    nhead=4,
    num_layers=2,
    dim_feedforward=512,
    dropout=0.2,
    target_tokens=240
)
context_model = SequenceGRUClassifier(
    epoch_model=encoder,
    hidden_size=128,
    num_layers=1,
    num_classes=5,
    bidirectional=False, # changed to unidirectional
)
context_model.to(device)

test_metrics_finetuned_unidir_l5_s1_dict = {}
if test_metrics_file_finetuned_unidir_l5_s1.exists():
    with open(test_metrics_file_finetuned_unidir_l5_s1, 'r') as f:
        test_metrics_finetuned_unidir_l5_s1_dict = json.load(f)
else:
    test_ds = BoasSequenceDataset(
        subjects=test_nights,
        mode='headband',
        seq_len=5,
        stride=5, # non-overlapping sequences
        transform_hb=None,
        preprocess_config=preprocess_config,
        use_cache=True
    )
    test_dl = DataLoader(
        test_ds,
        batch_size=64,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        drop_last=False,
        persistent_workers=False
    )
    for name in names_finetuned_unidir_l5_s1:
        checkpoint_path = CHECKPOINT_LEOMED_DIR / name / "best_model.pt"
        checkpoint_dict = torch.load(checkpoint_path, map_location='cpu')
        context_model.load_state_dict(checkpoint_dict['model_state_dict'])
        context_model.to(device)

        test_acc, test_mf1, test_perclass_f1 = forwardpass_testset(context_model, test_dl, device)
        test_metrics_finetuned_unidir_l5_s1_dict[name] = {
            'test_acc': test_acc,
            'test_mf1': test_mf1,
            'test_perclass_f1': test_perclass_f1.tolist()
        }
    with open(test_metrics_file_finetuned_unidir_l5_s1, 'w') as f:
        json.dump(test_metrics_finetuned_unidir_l5_s1_dict, f, indent=4)

c:\Users\leand\anaconda3\envs\sem-proj-gpu\lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(


Loading sub-1...
  ✓ Loaded sub-1 (headband) from cache
Loading sub-115...
  ✓ Loaded sub-115 (headband) from cache
Loading sub-116...
  ✓ Loaded sub-116 (headband) from cache
Loading sub-117...
  ✓ Loaded sub-117 (headband) from cache
Loading sub-118...
  ✓ Loaded sub-118 (headband) from cache
Loading sub-119...
  ✓ Loaded sub-119 (headband) from cache
Loading sub-120...
  ✓ Loaded sub-120 (headband) from cache
Loading sub-19...
  ✓ Loaded sub-19 (headband) from cache
Loading sub-2...
  ✓ Loaded sub-2 (headband) from cache
Loading sub-21...
  ✓ Loaded sub-21 (headband) from cache
Loading sub-24...
  ✓ Loaded sub-24 (headband) from cache
Loading sub-30...
  ✓ Loaded sub-30 (headband) from cache
Loading sub-34...
  ✓ Loaded sub-34 (headband) from cache
Loading sub-44...
  ✓ Loaded sub-44 (headband) from cache
Loading sub-46...
  ✓ Loaded sub-46 (headband) from cache
Loading sub-5...
  ✓ Loaded sub-5 (headband) from cache
Loading sub-56...
  ✓ Loaded sub-56 (headband) from cache
Loading 

C:\Users\leand\AppData\Local\Temp\ipykernel_77208\3257804741.py:44: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint_dict = torch.load(checkpoint_path, map_location=

In [6]:
# forward pass testset for fullysupervised model
encoder = SSLEpochTransformerConv1D_v2(
    d_model=128,
    nhead=4,
    num_layers=2,
    dim_feedforward=512,
    dropout=0.2,
    target_tokens=240
)
context_model = SequenceGRUClassifier(
    epoch_model=encoder,
    hidden_size=128,
    num_layers=1,
    num_classes=5,
    bidirectional=False, # changed to unidirectional
)
context_model.to(device)

test_metrics_fullysupervised_unidir_l5_s1_dict = {}
if test_metrics_file_fullysupervised_unidir_l5_s1.exists():
    with open(test_metrics_file_fullysupervised_unidir_l5_s1, 'r') as f:
        test_metrics_fullysupervised_unidir_l5_s1_dict = json.load(f)
else:
    test_ds = BoasSequenceDataset(
        subjects=test_nights,
        mode='headband',
        seq_len=5,
        stride=5, # non-overlapping sequences
        transform_hb=None,
        preprocess_config=preprocess_config,
        use_cache=True
    )
    test_dl = DataLoader(
        test_ds,
        batch_size=64,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        drop_last=False,
        persistent_workers=False
    )
    for name in names_fullysupervised_unidir_l5_s1:
        checkpoint_path = CHECKPOINT_LEOMED_DIR / name / "best_model.pt"
        checkpoint_dict = torch.load(checkpoint_path, map_location='cpu')
        context_model.load_state_dict(checkpoint_dict['model_state_dict'])
        context_model.to(device)

        test_acc, test_mf1, test_perclass_f1 = forwardpass_testset(context_model, test_dl, device)
        test_metrics_fullysupervised_unidir_l5_s1_dict[name] = {
            'test_acc': test_acc,
            'test_mf1': test_mf1,
            'test_perclass_f1': test_perclass_f1.tolist()
        }
    with open(test_metrics_file_fullysupervised_unidir_l5_s1, 'w') as f:
        json.dump(test_metrics_fullysupervised_unidir_l5_s1_dict, f, indent=4)

c:\Users\leand\anaconda3\envs\sem-proj-gpu\lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(


Loading sub-1...
  ✓ Loaded sub-1 (headband) from cache
Loading sub-115...
  ✓ Loaded sub-115 (headband) from cache
Loading sub-116...
  ✓ Loaded sub-116 (headband) from cache
Loading sub-117...
  ✓ Loaded sub-117 (headband) from cache
Loading sub-118...
  ✓ Loaded sub-118 (headband) from cache
Loading sub-119...
  ✓ Loaded sub-119 (headband) from cache
Loading sub-120...
  ✓ Loaded sub-120 (headband) from cache
Loading sub-19...
  ✓ Loaded sub-19 (headband) from cache
Loading sub-2...
  ✓ Loaded sub-2 (headband) from cache
Loading sub-21...
  ✓ Loaded sub-21 (headband) from cache
Loading sub-24...
  ✓ Loaded sub-24 (headband) from cache
Loading sub-30...
  ✓ Loaded sub-30 (headband) from cache
Loading sub-34...
  ✓ Loaded sub-34 (headband) from cache
Loading sub-44...
  ✓ Loaded sub-44 (headband) from cache
Loading sub-46...
  ✓ Loaded sub-46 (headband) from cache
Loading sub-5...
  ✓ Loaded sub-5 (headband) from cache
Loading sub-56...
  ✓ Loaded sub-56 (headband) from cache
Loading 

C:\Users\leand\AppData\Local\Temp\ipykernel_77208\1471394648.py:44: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint_dict = torch.load(checkpoint_path, map_location=

In [7]:
# plot test performance curves (varying data fraction)
mf1_finetuned_unidir_l5_s1 = []
mf1_fullysuperv_unidir_l5_s1 = []
for key in test_metrics_finetuned_unidir_l5_s1_dict.keys():
    nested_dict = test_metrics_finetuned_unidir_l5_s1_dict[key]
    mf1_score = nested_dict['test_mf1']
    mf1_finetuned_unidir_l5_s1.append(mf1_score)
for key in test_metrics_fullysupervised_unidir_l5_s1_dict.keys():
    nested_dict = test_metrics_fullysupervised_unidir_l5_s1_dict[key]
    mf1_score = nested_dict['test_mf1']
    mf1_fullysuperv_unidir_l5_s1.append(mf1_score)

p = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
p = [x * 100 for x in p]  # convert to percentage
plt.figure(figsize=(10, 6))
plt.plot(p, mf1_finetuned_unidir_l5_s1, marker='x', linewidth=2, markersize=8, label='SSL Pretrain + FT', color='green')
plt.plot(p, mf1_fullysuperv_unidir_l5_s1, marker='o', linewidth=2, markersize=8, label='Supervised from Scratch', color='red')
plt.xlabel('\nPercentage of labeled training data (%)', fontsize=18)
plt.ylabel('Macro F1 Score\n', fontsize=18)
plt.legend(fontsize=16)
plt.tick_params(axis='both', labelsize=16)
plt.tight_layout()
plt.grid(True)
# plt.show()
plt.savefig(TARGET_DIR / 'ctxsensitive_test_forward_pass_finetuning_vs_fullysupervised_varying_p_unidir_l5_s1_extra.pdf', dpi=300, bbox_inches='tight')

In [8]:
w_finetuned_unidir_l5_s1, n1_finetuned_unidir_l5_s1, n2_finetuned_unidir_l5_s1, n3_finetuned_unidir_l5_s1, rem_finetuned_unidir_l5_s1 = [], [], [], [], []
w_fullysuperv_unidir_l5_s1, n1_fullysuperv_unidir_l5_s1, n2_fullysuperv_unidir_l5_s1, n3_fullysuperv_unidir_l5_s1, rem_fullysuperv_unidir_l5_s1 = [], [], [], [], []
for key in test_metrics_finetuned_unidir_l5_s1_dict.keys():
    nested_dict = test_metrics_finetuned_unidir_l5_s1_dict[key]
    per_class_f1 = nested_dict['test_perclass_f1']
    w_finetuned_unidir_l5_s1.append(per_class_f1[0])
    n1_finetuned_unidir_l5_s1.append(per_class_f1[1])
    n2_finetuned_unidir_l5_s1.append(per_class_f1[2])
    n3_finetuned_unidir_l5_s1.append(per_class_f1[3])
    rem_finetuned_unidir_l5_s1.append(per_class_f1[4])
for key in test_metrics_fullysupervised_unidir_l5_s1_dict.keys():
    nested_dict = test_metrics_fullysupervised_unidir_l5_s1_dict[key]
    per_class_f1 = nested_dict['test_perclass_f1']
    w_fullysuperv_unidir_l5_s1.append(per_class_f1[0])
    n1_fullysuperv_unidir_l5_s1.append(per_class_f1[1])
    n2_fullysuperv_unidir_l5_s1.append(per_class_f1[2])
    n3_fullysuperv_unidir_l5_s1.append(per_class_f1[3])
    rem_fullysuperv_unidir_l5_s1.append(per_class_f1[4])


# plot 2x3 grid
p = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
p = [x * 100 for x in p]  # convert to percentage
fig, axes = plt.subplots(3, 2, figsize=(12, 14))
axes_flat = axes.flatten()
stages = ['Wake', 'N1', 'N2', 'N3', 'REM']
finetuned_data = [w_finetuned_unidir_l5_s1, n1_finetuned_unidir_l5_s1, n2_finetuned_unidir_l5_s1, n3_finetuned_unidir_l5_s1, rem_finetuned_unidir_l5_s1]
fullysuperv_data = [w_fullysuperv_unidir_l5_s1, n1_fullysuperv_unidir_l5_s1, n2_fullysuperv_unidir_l5_s1, n3_fullysuperv_unidir_l5_s1, rem_fullysuperv_unidir_l5_s1]
for idx, (ax, stage, fine, fully) in enumerate(zip(axes_flat[:5], stages, finetuned_data, fullysuperv_data)):
    ax.set_title(f'{stage}', fontsize=18, fontweight='bold')
    ax.plot(p, fine, marker='x', linewidth=2, markersize=7, color='green', label='SSL Pretrain + FT')
    ax.plot(p, fully, marker='o', linewidth=2, markersize=7, color='red', label='Supervised from Scratch')
    ax.set_xlabel('Percentage of labeled training data (%)', fontsize=18)
    ax.set_ylabel('F1 Score', fontsize=18)
    ax.tick_params(axis='both', labelsize=16)
    ax.grid(True)
    if idx == 0:
        ax.legend(fontsize=16)
axes_flat[5].axis('off')  # leave last subplot empty
plt.tight_layout()
plt.savefig(TARGET_DIR / 'finetuned_vs_fullysupervised_perclassf1_ctxsensitive_test_forward_pass_grid_unidir_l5_s1_3x2_extra.pdf', dpi=300, bbox_inches='tight')


BIDIR L10 s2

In [9]:
exp = "ctxsensitive_val_step_"
names_finetuned_bidir_l10_s2 = [exp + f"p{p}_ssl_finetuning_bidirTrue_L10_s2" for p in [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]]
names_fullysupervised_bidir_l10_s2 = [exp + f"p{p}_fully_supervised_bidirTrue_L10_s2" for p in [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]]
test_metrics_file_finetuned_bidir_l10_s2 = JSON_DIR / "ctxsensitive_finetuning_results_test_forward_pass_bidir_l10_s2_extra.json"
test_metrics_file_fullysupervised_bidir_l10_s2 = JSON_DIR / "ctxsensitive_fullysupervised_results_test_forward_pass_bidir_l10_s2_extra.json"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [10]:
# forward pass testset for finetuned model
encoder = SSLEpochTransformerConv1D_v2(
    d_model=128,
    nhead=4,
    num_layers=2,
    dim_feedforward=512,
    dropout=0.2,
    target_tokens=240
)
context_model = SequenceGRUClassifier(
    epoch_model=encoder,
    hidden_size=128,
    num_layers=1,
    num_classes=5,
    bidirectional=True,
)
context_model.to(device)

test_metrics_finetuned_bidir_l10_s2_dict = {}
if test_metrics_file_finetuned_bidir_l10_s2.exists():
    with open(test_metrics_file_finetuned_bidir_l10_s2, 'r') as f:
        test_metrics_finetuned_bidir_l10_s2_dict = json.load(f)
else:
    test_ds = BoasSequenceDataset(
        subjects=test_nights,
        mode='headband',
        seq_len=10,
        stride=10, # non-overlapping sequences
        transform_hb=None,
        preprocess_config=preprocess_config,
        use_cache=True
    )
    test_dl = DataLoader(
        test_ds,
        batch_size=64,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        drop_last=False,
        persistent_workers=False
    )
    for name in names_finetuned_bidir_l10_s2:
        checkpoint_path = CHECKPOINT_LEOMED_DIR / name / "best_model.pt"
        checkpoint_dict = torch.load(checkpoint_path, map_location='cpu')
        context_model.load_state_dict(checkpoint_dict['model_state_dict'])
        context_model.to(device)

        test_acc, test_mf1, test_perclass_f1 = forwardpass_testset(context_model, test_dl, device)
        test_metrics_finetuned_bidir_l10_s2_dict[name] = {
            'test_acc': test_acc,
            'test_mf1': test_mf1,
            'test_perclass_f1': test_perclass_f1.tolist()
        }
    with open(test_metrics_file_finetuned_bidir_l10_s2, 'w') as f:
        json.dump(test_metrics_finetuned_bidir_l10_s2_dict, f, indent=4)

c:\Users\leand\anaconda3\envs\sem-proj-gpu\lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(


Loading sub-1...
  ✓ Loaded sub-1 (headband) from cache
Loading sub-115...
  ✓ Loaded sub-115 (headband) from cache
Loading sub-116...
  ✓ Loaded sub-116 (headband) from cache
Loading sub-117...
  ✓ Loaded sub-117 (headband) from cache
Loading sub-118...
  ✓ Loaded sub-118 (headband) from cache
Loading sub-119...
  ✓ Loaded sub-119 (headband) from cache
Loading sub-120...
  ✓ Loaded sub-120 (headband) from cache
Loading sub-19...
  ✓ Loaded sub-19 (headband) from cache
Loading sub-2...
  ✓ Loaded sub-2 (headband) from cache
Loading sub-21...
  ✓ Loaded sub-21 (headband) from cache
Loading sub-24...
  ✓ Loaded sub-24 (headband) from cache
Loading sub-30...
  ✓ Loaded sub-30 (headband) from cache
Loading sub-34...
  ✓ Loaded sub-34 (headband) from cache
Loading sub-44...
  ✓ Loaded sub-44 (headband) from cache
Loading sub-46...
  ✓ Loaded sub-46 (headband) from cache
Loading sub-5...
  ✓ Loaded sub-5 (headband) from cache
Loading sub-56...
  ✓ Loaded sub-56 (headband) from cache
Loading 

C:\Users\leand\AppData\Local\Temp\ipykernel_77208\2270259388.py:44: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint_dict = torch.load(checkpoint_path, map_location=

In [11]:
# forard pass testset for fully supervised model
encoder = SSLEpochTransformerConv1D_v2(
    d_model=128,
    nhead=4,
    num_layers=2,
    dim_feedforward=512,
    dropout=0.2,
    target_tokens=240
)
context_model = SequenceGRUClassifier(
    epoch_model=encoder,
    hidden_size=128,
    num_layers=1,
    num_classes=5,
    bidirectional=True,
)
context_model.to(device)

test_metrics_fullysupervised_bidir_l10_s2_dict = {}
if test_metrics_file_fullysupervised_bidir_l10_s2.exists():
    with open(test_metrics_file_fullysupervised_bidir_l10_s2, 'r') as f:
        test_metrics_fullysupervised_bidir_l10_s2_dict = json.load(f)
else:
    test_ds = BoasSequenceDataset(
        subjects=test_nights,
        mode='headband',
        seq_len=10,
        stride=10, # non-overlapping sequences
        transform_hb=None,
        preprocess_config=preprocess_config,
        use_cache=True
    )
    test_dl = DataLoader(
        test_ds,
        batch_size=64,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        drop_last=False,
        persistent_workers=False
    )
    for name in names_fullysupervised_bidir_l10_s2:
        checkpoint_path = CHECKPOINT_LEOMED_DIR / name / "best_model.pt"
        checkpoint_dict = torch.load(checkpoint_path, map_location='cpu')
        context_model.load_state_dict(checkpoint_dict['model_state_dict'])
        context_model.to(device)

        test_acc, test_mf1, test_perclass_f1 = forwardpass_testset(context_model, test_dl, device)
        test_metrics_fullysupervised_bidir_l10_s2_dict[name] = {
            'test_acc': test_acc,
            'test_mf1': test_mf1,
            'test_perclass_f1': test_perclass_f1.tolist()
        }
    with open(test_metrics_file_fullysupervised_bidir_l10_s2, 'w') as f:
        json.dump(test_metrics_fullysupervised_bidir_l10_s2_dict, f, indent=4)

c:\Users\leand\anaconda3\envs\sem-proj-gpu\lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(


Loading sub-1...
  ✓ Loaded sub-1 (headband) from cache
Loading sub-115...
  ✓ Loaded sub-115 (headband) from cache
Loading sub-116...
  ✓ Loaded sub-116 (headband) from cache
Loading sub-117...
  ✓ Loaded sub-117 (headband) from cache
Loading sub-118...
  ✓ Loaded sub-118 (headband) from cache
Loading sub-119...
  ✓ Loaded sub-119 (headband) from cache
Loading sub-120...
  ✓ Loaded sub-120 (headband) from cache
Loading sub-19...
  ✓ Loaded sub-19 (headband) from cache
Loading sub-2...
  ✓ Loaded sub-2 (headband) from cache
Loading sub-21...
  ✓ Loaded sub-21 (headband) from cache
Loading sub-24...
  ✓ Loaded sub-24 (headband) from cache
Loading sub-30...
  ✓ Loaded sub-30 (headband) from cache
Loading sub-34...
  ✓ Loaded sub-34 (headband) from cache
Loading sub-44...
  ✓ Loaded sub-44 (headband) from cache
Loading sub-46...
  ✓ Loaded sub-46 (headband) from cache
Loading sub-5...
  ✓ Loaded sub-5 (headband) from cache
Loading sub-56...
  ✓ Loaded sub-56 (headband) from cache
Loading 

C:\Users\leand\AppData\Local\Temp\ipykernel_77208\2875535456.py:44: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint_dict = torch.load(checkpoint_path, map_location=

In [12]:
# plot test performance curves (varying data fraction)
mf1_finetuned_bidir_l10_s2 = []
mf1_fullysuperv_bidir_l10_s2 = []
for key in test_metrics_finetuned_bidir_l10_s2_dict.keys():
    nested_dict = test_metrics_finetuned_bidir_l10_s2_dict[key]
    mf1_score = nested_dict['test_mf1']
    mf1_finetuned_bidir_l10_s2.append(mf1_score)
for key in test_metrics_fullysupervised_bidir_l10_s2_dict.keys():
    nested_dict = test_metrics_fullysupervised_bidir_l10_s2_dict[key]
    mf1_score = nested_dict['test_mf1']
    mf1_fullysuperv_bidir_l10_s2.append(mf1_score)
p = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
p = [x * 100 for x in p]  # convert to percentage
plt.figure(figsize=(10, 6))
plt.plot(p, mf1_finetuned_bidir_l10_s2, marker='x', linewidth=2, markersize=8, label='SSL Pretrain + FT', color='green')
plt.plot(p, mf1_fullysuperv_bidir_l10_s2, marker='o', linewidth=2, markersize=8, label='Supervised from Scratch', color='red')
plt.xlabel('\nPercentage of labeled training data (%)', fontsize=18)
plt.ylabel('Macro F1 Score\n', fontsize=18)
plt.legend(fontsize=16)
plt.tick_params(axis='both', labelsize=16)
plt.tight_layout()
plt.grid(True)
# plt.show()
plt.savefig(TARGET_DIR / 'ctxsensitive_test_forward_pass_finetuning_vs_fullysupervised_varying_p_bidir_l10_s2_extra.pdf', dpi=300, bbox_inches='tight')

In [13]:
w_finetuned_bidir_l10_s2, n1_finetuned_bidir_l10_s2, n2_finetuned_bidir_l10_s2, n3_finetuned_bidir_l10_s2, rem_finetuned_bidir_l10_s2 = [], [], [], [], []
w_fullysuperv_bidir_l10_s2, n1_fullysuperv_bidir_l10_s2, n2_fullysuperv_bidir_l10_s2, n3_fullysuperv_bidir_l10_s2, rem_fullysuperv_bidir_l10_s2 = [], [], [], [], []
for key in test_metrics_finetuned_bidir_l10_s2_dict.keys():
    nested_dict = test_metrics_finetuned_bidir_l10_s2_dict[key]
    per_class_f1 = nested_dict['test_perclass_f1']
    w_finetuned_bidir_l10_s2.append(per_class_f1[0])
    n1_finetuned_bidir_l10_s2.append(per_class_f1[1])
    n2_finetuned_bidir_l10_s2.append(per_class_f1[2])
    n3_finetuned_bidir_l10_s2.append(per_class_f1[3])
    rem_finetuned_bidir_l10_s2.append(per_class_f1[4])
for key in test_metrics_fullysupervised_bidir_l10_s2_dict.keys():
    nested_dict = test_metrics_fullysupervised_bidir_l10_s2_dict[key]
    per_class_f1 = nested_dict['test_perclass_f1']
    w_fullysuperv_bidir_l10_s2.append(per_class_f1[0])
    n1_fullysuperv_bidir_l10_s2.append(per_class_f1[1])
    n2_fullysuperv_bidir_l10_s2.append(per_class_f1[2])
    n3_fullysuperv_bidir_l10_s2.append(per_class_f1[3])
    rem_fullysuperv_bidir_l10_s2.append(per_class_f1[4])


# plot 2x3 grid
p = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
p = [x * 100 for x in p]  # convert to percentage
fig, axes = plt.subplots(3, 2, figsize=(12, 14))
axes_flat = axes.flatten()
stages = ['Wake', 'N1', 'N2', 'N3', 'REM']
finetuned_data = [w_finetuned_bidir_l10_s2, n1_finetuned_bidir_l10_s2, n2_finetuned_bidir_l10_s2, n3_finetuned_bidir_l10_s2, rem_finetuned_bidir_l10_s2]
fullysuperv_data = [w_fullysuperv_bidir_l10_s2, n1_fullysuperv_bidir_l10_s2, n2_fullysuperv_bidir_l10_s2, n3_fullysuperv_bidir_l10_s2, rem_fullysuperv_bidir_l10_s2]
for idx, (ax, stage, fine, fully) in enumerate(zip(axes_flat[:5], stages, finetuned_data, fullysuperv_data)):
    ax.set_title(f'{stage}', fontsize=18, fontweight='bold')
    ax.plot(p, fine, marker='x', linewidth=2, markersize=7, color='green', label='SSL Pretrain + FT')
    ax.plot(p, fully, marker='o', linewidth=2, markersize=7, color='red', label='Supervised from Scratch')
    ax.set_xlabel('Percentage of labeled training data (%)', fontsize=18)
    ax.set_ylabel('F1 Score', fontsize=18)
    ax.tick_params(axis='both', labelsize=16)
    ax.grid(True)
    if idx == 0:
        ax.legend(fontsize=16)
axes_flat[5].axis('off')  # leave last subplot empty
plt.tight_layout()
plt.savefig(TARGET_DIR / 'finetuned_vs_fullysupervised_perclassf1_ctxsensitive_test_forward_pass_grid_bidir_l10_s2_3x2_extra.pdf', dpi=300, bbox_inches='tight')




how many parameters does the model have?

In [ ]:
encoder_params = sum(p.numel() for p in context_model.epoch_model.parameters())
total_params = sum(p.numel() for p in context_model.parameters())
print(f'Encoder parameters: {encoder_params}')
print(f'Total model parameters: {total_params}')
print(f'GRU then has {total_params - encoder_params} parameters')

Encoder parameters: 495968
Total model parameters: 595685
GRU then has 99717 parameters


plots for the models trained on 100% (to get visualization of bestperforming model)

start with bidirectional L10 s2 FINETUNED

In [14]:
# confusion matrix for one case (finetuned bidirectional L10 s2, p=1.0)
case_name = "ctxsensitive_val_step_p1.0_ssl_finetuning_bidirTrue_L10_s2"
checkpoint_path = CHECKPOINT_LEOMED_DIR / case_name / "best_model.pt"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

encoder = SSLEpochTransformerConv1D_v2(
    d_model=128,
    nhead=4,
    num_layers=2,
    dim_feedforward=512,
    dropout=0.2,
    target_tokens=240
)
context_model = SequenceGRUClassifier(
    epoch_model=encoder,
    hidden_size=128,
    num_layers=1,
    num_classes=5,
    bidirectional=True,
)
context_model.to(device)

test_ds = BoasSequenceDataset(
    subjects=test_nights,
    mode='headband',
    seq_len=10,
    stride=10,  # non-overlapping sequences
    transform_hb=None,
    preprocess_config=preprocess_config,
    use_cache=True
)
test_dl = DataLoader(
    test_ds,
    batch_size=64,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    drop_last=False,
    persistent_workers=False
)

checkpoint_dict = torch.load(checkpoint_path, map_location='cpu')
context_model.load_state_dict(checkpoint_dict['model_state_dict'])
context_model.to(device)

_, _, _, y_true, y_pred = forwardpass_testset(context_model, test_dl, device, return_preds=True)

labels = ["Wake", "N1", "N2", "N3", "REM"]
label_ids = list(range(len(labels)))

cm_counts = confusion_matrix(y_true, y_pred, labels=label_ids)

# --- normalize per TRUE class (row-normalized) ---
row_sums = cm_counts.sum(axis=1, keepdims=True)
cm_norm = np.divide(cm_counts, row_sums, out=np.zeros_like(cm_counts, dtype=float), where=row_sums != 0)

# Optional: show both percent and count in each cell
annot = np.empty_like(cm_counts, dtype=object)
for i in range(cm_counts.shape[0]):
    for j in range(cm_counts.shape[1]):
        annot[i, j] = f"{cm_norm[i, j]*100:.1f}%\n({cm_counts[i, j]})"

plt.figure(figsize=(7, 6))
ax = sns.heatmap(
    cm_norm,
    annot=annot,          # comment this line and uncomment the next two lines if you want only %
    fmt="",               # needed because annot is strings
    # annot=True,
    # fmt=".2f",
    cmap="cividis",       # perceptually uniform + colorblind-friendly
    vmin=0, vmax=1,       # fixed scale for normalized values
    xticklabels=labels,
    yticklabels=labels,
    cbar_kws={"label": "Recall"}  # can change to "Recall"
)

ax.set_xlabel("Predicted", fontsize=18)
ax.set_ylabel("True", fontsize=18)
# ax.set_title("Confusion Matrix (row-normalized): Finetuned Bidir L10 s2 (p=1.0)", fontsize=16)
plt.tight_layout()

out_path = TARGET_DIR / "confmat_testset_finetuned_bidir_l10_s2_p1.0_rowNorm_extra.pdf"
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()


c:\Users\leand\anaconda3\envs\sem-proj-gpu\lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(


Loading sub-1...
  ✓ Loaded sub-1 (headband) from cache
Loading sub-115...
  ✓ Loaded sub-115 (headband) from cache
Loading sub-116...
  ✓ Loaded sub-116 (headband) from cache
Loading sub-117...
  ✓ Loaded sub-117 (headband) from cache
Loading sub-118...
  ✓ Loaded sub-118 (headband) from cache
Loading sub-119...
  ✓ Loaded sub-119 (headband) from cache
Loading sub-120...
  ✓ Loaded sub-120 (headband) from cache
Loading sub-19...
  ✓ Loaded sub-19 (headband) from cache
Loading sub-2...
  ✓ Loaded sub-2 (headband) from cache
Loading sub-21...
  ✓ Loaded sub-21 (headband) from cache
Loading sub-24...
  ✓ Loaded sub-24 (headband) from cache
Loading sub-30...
  ✓ Loaded sub-30 (headband) from cache
Loading sub-34...
  ✓ Loaded sub-34 (headband) from cache
Loading sub-44...
  ✓ Loaded sub-44 (headband) from cache
Loading sub-46...
  ✓ Loaded sub-46 (headband) from cache
Loading sub-5...
  ✓ Loaded sub-5 (headband) from cache
Loading sub-56...
  ✓ Loaded sub-56 (headband) from cache
Loading 

C:\Users\leand\AppData\Local\Temp\ipykernel_77208\4291190152.py:42: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint_dict = torch.load(checkpoint_path, map_location=

In [ ]:
with open(test_metrics_file_finetuned_bidir_l10_s2, 'r') as f:
    test_metrics_finetuned_bidir_l10_s2_dict = json.load(f)
test_metrics_finetuned_bidir_l10_s2_p1 = test_metrics_finetuned_bidir_l10_s2_dict['ctxsensitive_val_step_p1.0_ssl_finetuning_bidirTrue_L10_s2']
y = test_metrics_finetuned_bidir_l10_s2_p1['test_perclass_f1']
x = ["Wake", "N1", "N2", "N3", "REM"]
plt.figure(figsize=(10, 6))
plt.bar(x, y)
plt.xlabel('\nSleep Stage', fontsize=20)
plt.ylabel('F1 Score\n', fontsize=20)
plt.tick_params(axis='both', labelsize=16)
plt.ylim(0, 1)
plt.axhline(y=test_metrics_finetuned_bidir_l10_s2_p1['test_mf1'], color='r', linestyle='--', label='Macro F1 Score')
for i, f1 in enumerate(y):
    plt.text(i, f1 + 0.02, round(f1, 3), ha='center', fontsize=18)
plt.text(1, 0.79, round(test_metrics_finetuned_bidir_l10_s2_p1['test_mf1'], 3), ha='center', color='r', fontsize=18)
plt.legend(fontsize=12, bbox_to_anchor=(0.83, 1), loc='upper right')
plt.savefig(TARGET_DIR / 'finetuned_contextsensitive_test_forward_pass_bidir_l10_s2_p1.0_barplot.pdf', dpi=300, bbox_inches='tight')

continue with bidirectional L10 s2 fully supervised

In [ ]:
with open(test_metrics_file_fullysupervised_bidir_l10_s2, 'r') as f:
    test_metrics_fullysupervised_bidir_l10_s2_dict = json.load(f)
test_metrics_fullysupervised_bidir_l10_s2_p1 = test_metrics_fullysupervised_bidir_l10_s2_dict['ctxsensitive_val_step_p1.0_fully_supervised_bidirTrue_L10_s2']
y = test_metrics_fullysupervised_bidir_l10_s2_p1['test_perclass_f1']
x = ["Wake", "N1", "N2", "N3", "REM"]
plt.figure(figsize=(10, 6))
plt.bar(x, y)
plt.xlabel('\nSleep Stage', fontsize=20)
plt.ylabel('F1 Score\n', fontsize=20)
plt.tick_params(axis='both', labelsize=16)
plt.ylim(0, 1)
plt.axhline(y=test_metrics_fullysupervised_bidir_l10_s2_p1['test_mf1'], color='r', linestyle='--', label='Macro F1 Score')
for i, f1 in enumerate(y):
    plt.text(i, f1 + 0.02, round(f1, 3), ha='center', fontsize=18)
plt.text(1, 0.78, round(test_metrics_fullysupervised_bidir_l10_s2_p1['test_mf1'], 3), ha='center', color='r', fontsize=18)
plt.legend(fontsize=12, bbox_to_anchor=(0.83, 1), loc='upper right')
plt.savefig(TARGET_DIR / 'fullysupervised_contextsensitive_test_forward_pass_bidir_l10_s2_p1.0_barplot.pdf', dpi=300, bbox_inches='tight')

switch to unidirectional L5 s1 FINETUNED

In [ ]:
with open(test_metrics_file_finetuned_unidir_l5_s1, 'r') as f:
    test_metrics_finetuned_unidir_l5_s1_dict = json.load(f)
test_metrics_finetuned_unidir_l5_s1_p1 = test_metrics_finetuned_unidir_l5_s1_dict['ctxsensitive_val_step_p1.0_ssl_finetuning_bidirFalse_L5_s1']
y = test_metrics_finetuned_unidir_l5_s1_p1['test_perclass_f1']
x = ["Wake", "N1", "N2", "N3", "REM"]
plt.figure(figsize=(10, 6))
plt.bar(x, y)
plt.xlabel('\nSleep Stage', fontsize=20)
plt.ylabel('F1 Score\n', fontsize=20)
plt.tick_params(axis='both', labelsize=16)
plt.ylim(0, 1)
plt.axhline(y=test_metrics_finetuned_unidir_l5_s1_p1['test_mf1'], color='r', linestyle='--', label='Macro F1 Score')
for i, f1 in enumerate(y):
    plt.text(i, f1 + 0.02, round(f1, 3), ha='center', fontsize=18)
plt.text(1, 0.78, round(test_metrics_finetuned_unidir_l5_s1_p1['test_mf1'], 3), ha='center', color='r', fontsize=18)
plt.legend(fontsize=12)
plt.savefig(TARGET_DIR / 'finetuned_contextsensitive_test_forward_pass_unidir_l5_s1_p1.0_barplot.pdf', dpi=300, bbox_inches='tight')

continue with unidirectional L5 s1 fully supervised

In [ ]:
with open(test_metrics_file_fullysupervised_unidir_l5_s1, 'r') as f:
    test_metrics_fullysupervised_unidir_l5_s1_dict = json.load(f)
test_metrics_fullysupervised_unidir_l5_s1_p1 = test_metrics_fullysupervised_unidir_l5_s1_dict['ctxsensitive_val_step_p1.0_fully_supervised_bidirFalse_L5_s1']
y = test_metrics_fullysupervised_unidir_l5_s1_p1['test_perclass_f1']
x = ["Wake", "N1", "N2", "N3", "REM"]
plt.figure(figsize=(10, 6))
plt.bar(x, y)
plt.xlabel('\nSleep Stage', fontsize=20)
plt.ylabel('F1 Score\n', fontsize=20)
plt.tick_params(axis='both', labelsize=16)
plt.ylim(0, 1)
plt.axhline(y=test_metrics_fullysupervised_unidir_l5_s1_p1['test_mf1'], color='r', linestyle='--', label='Macro F1 Score')
for i, f1 in enumerate(y):
    plt.text(i, f1 + 0.02, round(f1, 3), ha='center', fontsize=18)
plt.text(1, 0.78, round(test_metrics_fullysupervised_unidir_l5_s1_p1['test_mf1'], 3), ha='center', color='r', fontsize=18)
plt.legend(fontsize=12)
plt.savefig(TARGET_DIR / 'fullysupervised_contextsensitive_test_forward_pass_unidir_l5_s1_p1.0_barplot.pdf', dpi=300, bbox_inches='tight')